# XML De-Identification Pipeline (HL7 CDA)

Handles two types of PHI in the CDA document:

| PHI Type | Strategy |
|---|---|
| Structured fields (name, DOB, SSN, phone, ZIP, address, dates) | **XML-aware rules** — direct tag/attribute targeting |
| Narrative `<text>` sections (free clinical text) | **NLP pipeline** — same ZeroShot NER as plain-text pipeline |

Same 5 rules apply:
- **Patient Name** → Patient ID
- **Doctor Name** → kept as-is
- **Service Dates** → Month + Year
- **DOB** → Age
- **ZIP** → first 3 chars + XX

In [ ]:
import json, os, re, copy
import xml.etree.ElementTree as ET
from datetime import datetime, date as date_type
import pandas as pd

XML_INPUT  = "file9.txt"
XML_OUTPUT = "file9_deid.xml"
EXCEL_OUT  = "XML_DeID_Results.xlsx"

print("Setup done.")


In [ ]:
# Spark NLP not used — Stage 2 uses pure-Python regex rules
# (JSL floating license supports only one active session at a time)


## Configuration

In [ ]:
# ── HL7 CDA default namespace ─────────────────────────────────────────────
NS  = "urn:hl7-org:v3"
NSM = {"cda": NS}  # for findall with prefix

def t(name):
    """Return fully-qualified tag name for the CDA namespace."""
    return f"{{{NS}}}{name}"

# ── Patient ID lookup ─────────────────────────────────────────────────────
PATIENT_ID_MAP = {
    "Myra Jones"  : "PT-00001",   # patient in file9.txt
    # Add more patients as needed
}

def get_patient_id(name: str) -> str:
    return PATIENT_ID_MAP.get(name.strip(), f"PT-{abs(hash(name)) % 90000 + 10000}")

print("Config ready. Patient map:", PATIENT_ID_MAP)

## Helper Functions

In [ ]:
# ── Date parsers ──────────────────────────────────────────────────────────

def yyyymmdd_to_month_year(value: str) -> str:
    """CDA date format YYYYMMDD[HH...] → 'September 2012'"""
    try:
        return datetime.strptime(value[:8], "%Y%m%d").strftime("%B %Y")
    except Exception:
        try:
            return datetime.strptime(value[:6], "%Y%m").strftime("%B %Y")
        except Exception:
            return value[:4] if len(value) >= 4 else "[DATE]"

def yyyymmdd_to_age(value: str) -> str:
    """CDA birthTime YYYYMMDD → '77 years old'"""
    try:
        dt = datetime.strptime(value[:8], "%Y%m%d")
        today = date_type.today()
        age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
        if 0 <= age <= 120:
            return f"{age} years old"
    except Exception:
        pass
    return "[DOB]"

def partial_zip(z: str) -> str:
    """97006 → 970XX"""
    z = z.strip()
    return z[:3] + "X" * (len(z) - 3) if len(z) > 3 else z

def iter_text(el):
    """Recursively yield all text content from an element."""
    texts = []
    if el.text:
        texts.append(el.text.strip())
    for child in el:
        texts.extend(iter_text(child))
        if child.tail:
            texts.append(child.tail.strip())
    return [t for t in texts if t]

# Quick tests
print(yyyymmdd_to_month_year("20120912000000-0000"))  # September 2012
print(yyyymmdd_to_age("19470501"))                    # 77 years old
print(partial_zip("97006"))                            # 970XX

## Stage 1 — Structural XML De-Identification
Targets known PHI locations in the CDA header using tag names and parent context.

In [ ]:
# Register default namespace so output doesn't get ugly ns0: prefixes
ET.register_namespace("", NS)
ET.register_namespace("xsi",   "http://www.w3.org/2001/XMLSchema-instance")
ET.register_namespace("sdtc",  "urn:hl7-org:sdtc")
ET.register_namespace("cda",   "urn:hl7-org:v3")

tree = ET.parse(XML_INPUT)
root = tree.getroot()

# Changelog — every change recorded here for Excel report
changelog = []   # {location, field_type, original, rule, deid}

def log(location, field_type, original, rule, deid):
    changelog.append({
        "XML Location"      : location,
        "Field Type"        : field_type,
        "Original Value"    : str(original),
        "Rule Applied"      : rule,
        "De-identified Value": str(deid),
    })

# ── 1. Patient name (under <patient><name>) ───────────────────────────────
for patient_el in root.iter(t("patient")):
    for name_el in patient_el.findall(t("name")):
        given_el  = name_el.find(t("given"))
        family_el = name_el.find(t("family"))
        if given_el is not None and family_el is not None:
            full_name  = f"{given_el.text} {family_el.text}"
            patient_id = get_patient_id(full_name)
            log("patient/name", "PATIENT_NAME", full_name, "→ Patient ID", patient_id)
            given_el.text  = patient_id
            family_el.text = ""

# ── 2. Doctor names (under <assignedPerson><name>) — keep as-is ───────────
for ap_el in root.iter(t("assignedPerson")):
    for name_el in ap_el.findall(t("name")):
        given_el  = name_el.find(t("given"))
        family_el = name_el.find(t("family"))
        if given_el is not None and family_el is not None:
            full_name = f"{given_el.text} {family_el.text}"
            log("assignedPerson/name", "DOCTOR_NAME", full_name, "kept as-is", full_name)

# ── 3. DOB — <birthTime value="YYYYMMDD"> ────────────────────────────────
for bt_el in root.iter(t("birthTime")):
    val = bt_el.get("value", "")
    if val:
        age_str = yyyymmdd_to_age(val)
        log("birthTime/@value", "DOB", val, "→ Age", age_str)
        bt_el.set("value", age_str)

# ── 4. SSN — <id root="2.16.840.1.113883.4.1"> ───────────────────────────
SSN_ROOT = "2.16.840.1.113883.4.1"
for id_el in root.iter(t("id")):
    if id_el.get("root") == SSN_ROOT:
        orig = id_el.get("extension", "")
        log("id[@root=SSN]/@extension", "SSN", orig, "→ [SSN]", "[SSN]")
        id_el.set("extension", "[SSN]")

# ── 5. Service dates — <effectiveTime value="YYYYMMDD">, <low>, <high> ────
DATE_TAGS = {t("effectiveTime"), t("low"), t("high"), t("time")}
for el in root.iter():
    if el.tag in DATE_TAGS:
        val = el.get("value", "")
        if val and len(val) >= 6 and val[:4].isdigit():
            month_year = yyyymmdd_to_month_year(val)
            log(f"{el.tag.split('}')[1]}/@value", "SERVICE_DATE", val, "→ Month+Year", month_year)
            el.set("value", month_year)

# ── 6. ZIP codes — <postalCode> ──────────────────────────────────────────
for pc_el in root.iter(t("postalCode")):
    if pc_el.text and pc_el.text.strip():
        orig = pc_el.text.strip()
        masked = partial_zip(orig)
        log("postalCode", "ZIP", orig, f"→ first 3 + XX", masked)
        pc_el.text = masked

# ── 7. Phone numbers — <telecom value="tel:..."> ─────────────────────────
for tc_el in root.iter(t("telecom")):
    val = tc_el.get("value", "")
    if val.startswith("tel:") or val.startswith("tel:"):
        log("telecom/@value", "PHONE", val, "→ [PHONE]", "tel:[PHONE]")
        tc_el.set("value", "tel:[PHONE]")

# ── 8. Street addresses — <streetAddressLine> ────────────────────────────
for sa_el in root.iter(t("streetAddressLine")):
    if sa_el.text and sa_el.text.strip():
        orig = sa_el.text.strip()
        log("streetAddressLine", "STREET", orig, "→ [STREET]", "[STREET]")
        sa_el.text = "[STREET]"

# ── 9. Related / associated persons (spouse, relatives) ──────────────────
for tag_name in ["relatedPerson", "associatedPerson", "informationRecipient"]:
    for rp_el in root.iter(t(tag_name)):
        for name_el in rp_el.findall(t("name")):
            given_el  = name_el.find(t("given"))
            family_el = name_el.find(t("family"))
            if given_el is not None and family_el is not None:
                full_name = f"{given_el.text} {family_el.text}"
                log(f"{tag_name}/name", "RELATED_PERSON", full_name, "→ [RELATED_PERSON]", "[RELATED_PERSON]")
                given_el.text  = "[RELATED"
                family_el.text = "PERSON]"

print(f"Structural pass complete. {len(changelog)} changes recorded.")

## Stage 2 — Regex Rules for Narrative `<text>` Sections
Applies targeted string-replacement rules to the free-text narrative content
without requiring the Spark NLP pipeline (no license dependency).


In [ ]:
# ── Pure-Python regex de-identification rules for narrative text ─────────
import re
from datetime import datetime, date as date_type

# Patient name variants (first/last, last/first, with titles)
_PATIENT_PATTERNS = [
    (re.compile(r'\bMyra\s+Jones\b',  re.I), 'PT-00001'),
]

# SSN pattern
_SSN_RE = re.compile(r'\b\d{3}-\d{2}-\d{4}\b')

# US phone patterns
_PHONE_RE = re.compile(
    r'(?:\(\d{3}\)\s*|\d{3}[-.])'  # (816) or 816-
    r'\d{3}[-.]\d{4}'
    r'(?:\s*(?:x|ext)\.?\s*\d+)?',
    re.I
)

# ZIP code patterns (5-digit or ZIP+4 or UK)
_ZIP_RE = re.compile(r'\b(\d{5})(?:-\d{4})?\b')

# YYYYMMDD pattern (common inside CDA <td> cells)
_YYYYMMDD_RE = re.compile(r'\b((?:19|20)\d{6})\b')

# Date formats commonly found in narrative (non-YYYYMMDD)
_DATE_FORMATS = [
    '%m/%d/%Y', '%d/%m/%Y', '%Y-%m-%d',
    '%B %d, %Y', '%b %d, %Y', '%d %B %Y', '%d %b %Y',
    '%B %d %Y', '%b %d %Y',
]

# DOB keywords followed by a date
_DOB_KW_RE = re.compile(
    r'(?:DOB|D\.O\.B\.?|Date\s+of\s+Birth|Born(?:\s+on)?|Birthday)[:\s]+'
    r'([\w,/\-\.]+(?:\s+\d{4})?)',
    re.I
)

# Service date keywords
_SVC_DATE_KW_RE = re.compile(
    r'(?:Admission|Discharge|Service|Appointment|Visit|Record|Encounter)\s+[Dd]ate[:\s]+'
    r'([\w,/\-\.]+(?:\s+\d{4})?)',
    re.I
)

# Standalone date pattern (MM/DD/YYYY, YYYY-MM-DD, etc.)
_STANDALONE_DATE_RE = re.compile(
    r'\b(?:\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}|\d{4}[/\-\.]\d{1,2}[/\-\.]\d{1,2})\b'
)

def _parse_date(s):
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s.strip())
    for fmt in _DATE_FORMATS:
        try: return datetime.strptime(s, fmt)
        except ValueError: pass
    return None

def _dob_to_age(s):
    dt = _parse_date(s)
    if dt:
        today = date_type.today()
        age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
        if 0 <= age <= 120:
            return f"{age} years old"
    return '[DOB]'

def _date_to_month_year(s):
    dt = _parse_date(s)
    if dt: return dt.strftime('%B %Y')
    m = re.search(r'\b(19|20)\d{2}\b', s)
    return m.group(0) if m else '[DATE]'

def _partial_zip(m):
    z = m.group(0)
    return z[:3] + 'X' * (len(z) - 3)

def deid_narrative(text: str, section_idx: int) -> str:
    """Apply de-ID rules to a single narrative text string."""
    orig = text

    # 1. Patient name
    for pat, pid in _PATIENT_PATTERNS:
        text = pat.sub(pid, text)

    # 2. SSN
    text = _SSN_RE.sub('[SSN]', text)

    # 3. Phone
    text = _PHONE_RE.sub('[PHONE]', text)

    # 4. DOB keyword + date
    def _replace_dob_kw(m):
        age = _dob_to_age(m.group(1))
        return m.group(0).replace(m.group(1), age)
    text = _DOB_KW_RE.sub(_replace_dob_kw, text)

    # 5. Service date keyword + date
    def _replace_svc_kw(m):
        my = _date_to_month_year(m.group(1))
        return m.group(0).replace(m.group(1), my)
    text = _SVC_DATE_KW_RE.sub(_replace_svc_kw, text)

    # 6. Remaining standalone dates → month+year
    def _replace_date(m):
        return _date_to_month_year(m.group(0))
    text = _STANDALONE_DATE_RE.sub(_replace_date, text)

    # 7. YYYYMMDD dates in narrative table cells
    def _replace_yyyymmdd(m):
        val = m.group(1)
        try:
            from datetime import datetime
            dt = datetime.strptime(val, '%Y%m%d')
            return dt.strftime('%B %Y')
        except Exception:
            return val
    text = _YYYYMMDD_RE.sub(_replace_yyyymmdd, text)

    # 8. ZIP
    text = _ZIP_RE.sub(_partial_zip, text)

    if text != orig:
        log(f'<text> section #{section_idx}', 'NARRATIVE_TEXT',
            orig[:80] + ('...' if len(orig) > 80 else ''),
            'regex rules',
            text[:80] + ('...' if len(text) > 80 else ''))
    return text

print('Narrative regex rules ready.')


In [ ]:
# ── Apply regex rules to all narrative <text> sections ───────────────────
processed = 0
for idx, text_el in enumerate(root.iter(t('text'))):
    # Collect the full text content of this element (including tail text of children)
    flat_text = ' '.join(iter_text(text_el)).strip()
    if not flat_text:
        continue
    deid_text = deid_narrative(flat_text, idx)
    if deid_text != flat_text:
        # Replace element text content with de-identified version
        for child in list(text_el):
            text_el.remove(child)
        text_el.text = deid_text
        processed += 1

print(f'Narrative sections processed: {processed}')


In [ ]:
# (Stage 2 complete — narrative sections updated in-place above)
print('Stage 2 complete.')


In [ ]:
# ── Save de-identified XML ────────────────────────────────────────────────
tree.write(XML_OUTPUT, encoding="unicode", xml_declaration=True)
print(f"De-identified XML saved: {XML_OUTPUT}")

# Quick peek at patient section in output
with open(XML_OUTPUT) as f:
    content = f.read()

# Show patient block
start = content.find("<patientRole")
end   = content.find("</patientRole>") + len("</patientRole>")
print("\n── Patient section preview ──")
print(content[start:end][:1200])

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

changes_df = pd.DataFrame(changelog)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "De-ID Changes"

header_fill  = PatternFill("solid", fgColor="1F4E79")
header_font  = Font(bold=True, color="FFFFFF", size=11)
alt_fill     = PatternFill("solid", fgColor="DEEAF1")
orig_fill    = PatternFill("solid", fgColor="FCE4D6")   # orange — original
deid_fill    = PatternFill("solid", fgColor="E2EFDA")   # green  — de-identified
wrap_align   = Alignment(wrap_text=True, vertical="top")
center_align = Alignment(horizontal="center", vertical="top")
thin_border  = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

headers    = ["#", "XML Location", "Field Type", "Original Value", "Rule Applied", "De-identified Value"]
col_widths = [4,   30,             18,            30,               22,             30]

for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
    cell = ws.cell(row=1, column=ci, value=h)
    cell.font = header_font; cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = thin_border
    ws.column_dimensions[get_column_letter(ci)].width = w
ws.row_dimensions[1].height = 22

for ri, row in changes_df.iterrows():
    er = ri + 2
    is_alt = ri % 2 == 0
    vals = [ri+1, row["XML Location"], row["Field Type"],
            row["Original Value"], row["Rule Applied"], row["De-identified Value"]]
    for ci, val in enumerate(vals, 1):
        cell = ws.cell(row=er, column=ci, value=str(val) if val else "")
        cell.alignment = center_align if ci == 1 else wrap_align
        cell.border    = thin_border
        if   ci == 4: cell.fill = orig_fill
        elif ci == 6: cell.fill = deid_fill
        elif is_alt:  cell.fill = alt_fill
    ws.row_dimensions[er].height = 45

ws.freeze_panes = "A2"

# Summary sheet
ws2 = wb.create_sheet("Summary")
ws2.column_dimensions["A"].width = 25
ws2.column_dimensions["B"].width = 40
summary = [
    ("Input file",     XML_INPUT),
    ("Output file",    XML_OUTPUT),
    ("Document type",  "HL7 CDA (Clinical Document Architecture)"),
    ("Total changes",  str(len(changelog))),
    ("", ""),
    ("Stage 1", "XML-aware structural rules"),
    ("  PATIENT_NAME",   "→ Patient ID from lookup"),
    ("  DOCTOR_NAME",    "→ kept as-is"),
    ("  DOB",            "→ Age (e.g. 77 years old)"),
    ("  SERVICE_DATE",   "→ Month + Year"),
    ("  SSN",            "→ [SSN]"),
    ("  PHONE",          "→ tel:[PHONE]"),
    ("  STREET",         "→ [STREET]"),
    ("  ZIP",            "→ first 3 + XX"),
    ("  RELATED_PERSON", "→ [RELATED PERSON]"),
    ("", ""),
    ("Stage 2", "NLP pipeline on narrative <text> sections"),
    ("  Model", "zeroshot_ner_deid_subentity_docwise_medium"),
]
for r, (k, v) in enumerate(summary, 1):
    ws2.cell(row=r, column=1, value=k).font = Font(bold=True)
    ws2.cell(row=r, column=2, value=v)

wb.save(EXCEL_OUT)
print(f"\nExcel report saved: {EXCEL_OUT}")
print(f"Total changes logged: {len(changelog)}")
changes_df